# 100K Row Scaling Benchmark

- **Subsample workflow**: 5K init -> batch insert -> full 100K inference
- **Data connector roundtrip**: `save_npy` / `load_npy_mmap` at 100K scale

Min VRAM: 16GB.

## 1. Setup

In [ ]:
!nvidia-smi

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

BRANCH = "main"  # @param {type:"string"}

try:
    import crosscat

    print(f"crosscat {crosscat.__version__} already installed")
except ImportError:
    if "COLAB_RELEASE_TAG" in os.environ:
        WORKDIR = "/content/jaxcross"
    elif Path("/kaggle/working").exists():
        WORKDIR = "/kaggle/working/jaxcross"
    else:
        WORKDIR = str(Path.home() / "jaxcross")
    if not Path(WORKDIR).exists():
        subprocess.run(
            ["git", "clone", "https://github.com/sambhal-labs/jaxcross.git", WORKDIR],
            check=True,
        )
    subprocess.run(["git", "fetch", "origin"], cwd=WORKDIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=WORKDIR, check=True)
    subprocess.run(["git", "pull", "origin", BRANCH], cwd=WORKDIR, check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", WORKDIR, "--no-deps", "-q"],
        check=True,
    )
    os.chdir(WORKDIR)

In [ ]:
import json
import tempfile
import time

import jax
import jax.numpy as jnp
import numpy as np

from benchmarks.utils import detect_platform, make_benchmark_data
from crosscat import (
    initialize,
    load_npy_mmap,
    pack_state,
    packed_gibbs_sweep,
    packed_insert_rows,
    save_npy,
    suggest_max_clusters,
)

platform = detect_platform()
print(f"Platform: {platform['platform']}, Backend: {platform['backend']}")
print(f"GPUs: {platform['n_gpus']}x {platform['gpu_names']}")
assert jax.default_backend() in ("gpu", "tpu"), "This notebook requires GPU/TPU runtime!"

## 2. Data Connector Roundtrip

Test `save_npy` / `load_npy_mmap` at 100K scale.

In [ ]:
print("--- Data Connector Roundtrip: 100,000 x 20 ---")
k1 = jax.random.fold_in(jax.random.key(42), 1)
data, col_types = make_benchmark_data(k1, 100_000, 20)
col_names = [f"col_{j}" for j in range(20)]

from pathlib import Path

tmp_dir = Path(tempfile.mkdtemp(prefix="jaxcross_bench_"))
npy_path = tmp_dir / "test_100k.npy"

t0 = time.perf_counter()
save_npy(npy_path, data, column_names=col_names)
save_time = time.perf_counter() - t0
file_size_mb = npy_path.stat().st_size / (1024 * 1024)
print(f"  save_npy: {save_time:.2f}s ({file_size_mb:.1f} MB)")

t0 = time.perf_counter()
loaded_data, loaded_names = load_npy_mmap(npy_path)
load_time = time.perf_counter() - t0
print(f"  load_npy_mmap: {load_time:.2f}s")

assert loaded_data.shape == data.shape
assert np.allclose(loaded_data, np.asarray(data), equal_nan=True)
assert loaded_names == col_names
print("  roundtrip verified OK")

npy_path.unlink(missing_ok=True)
npy_path.with_suffix(".json").unlink(missing_ok=True)

## 3. Subsample Workflow (5K -> 100K)

Init on 5K rows, batch insert 95K, then sweep on full 100K.

In [ ]:
print("--- Subsample Workflow: 100,000 rows x 20 cols ---")
k1, k2, k3, k4, k5 = jax.random.split(jax.random.fold_in(jax.random.key(42), 2), 5)
data, col_types = make_benchmark_data(k1, 100_000, 20)
max_k = suggest_max_clusters(100_000)
print(f"  data: {data.nbytes / (1024 * 1024):.1f} MB, max_clusters: {max_k}")

# Step 1: Subsample init
t0 = time.perf_counter()
result = initialize(k2, data, col_types, subsample_rows=5000)
sub_idx = result.subsample_idx
sub_data = data[sub_idx]
packed = pack_state(result.state, max_clusters=max_k)
print(f"  1. subsample init (5000 rows): {time.perf_counter() - t0:.2f}s")

# Step 2: Pre-sweeps
t0 = time.perf_counter()
packed = packed_gibbs_sweep(k3, packed, sub_data, n_sweeps=5)
packed.column_assignments.block_until_ready()
print(f"  2. 5 pre-sweeps on subsample: {time.perf_counter() - t0:.2f}s")

# Step 3: Batch insert
remaining_mask = jnp.ones(100_000, dtype=bool).at[sub_idx].set(False)
remaining_idx = jnp.where(remaining_mask, size=100_000 - 5000)[0]
remaining_data = data[remaining_idx]
batch_size = 5000
n_batches = (remaining_data.shape[0] + batch_size - 1) // batch_size

t0 = time.perf_counter()
current_data = sub_data
for b in range(n_batches):
    batch = remaining_data[b * batch_size : (b + 1) * batch_size]
    kb = jax.random.fold_in(k4, b)
    packed, current_data = packed_insert_rows(kb, packed, current_data, batch)
    if (b + 1) % 5 == 0 or b == n_batches - 1:
        elapsed = time.perf_counter() - t0
        print(f"     batch {b + 1}/{n_batches}: {packed.n_rows} rows, {elapsed:.1f}s elapsed")
insert_time = time.perf_counter() - t0
print(f"  3. inserted {remaining_data.shape[0]} rows: {insert_time:.2f}s")

# Step 4: Post-sweeps
t0 = time.perf_counter()
packed = packed_gibbs_sweep(k5, packed, current_data, n_sweeps=3)
packed.column_assignments.block_until_ready()
post_time = time.perf_counter() - t0
per_sweep_full = post_time / 3
print(
    f"  4. 3 post-sweeps on full {packed.n_rows} rows: {post_time:.2f}s ({per_sweep_full:.2f}s/sweep)"
)

## 4. Save Results

In [ ]:
import shutil
from pathlib import Path

results_dir = Path("benchmarks/results/scaling")
results_dir.mkdir(parents=True, exist_ok=True)

results_100k = {
    "backend": platform["backend"],
    "device": str(jax.devices()[0]),
    "connectors": {"save_time": save_time, "load_time": load_time, "file_size_mb": file_size_mb},
    "workflow": {"per_sweep_full": per_sweep_full, "insert_time": insert_time},
}
with open(results_dir / "scaling_100k_results.json", "w") as f:
    json.dump(results_100k, f, indent=2)

print(f"Results saved to {results_dir / 'scaling_100k_results.json'}")
archive = Path("benchmarks/results/scaling_100k_results")
shutil.make_archive(str(archive), "gztar", ".", str(results_dir))
print(f"Archived to {archive}.tar.gz")